# Validación del flujo completo con el target nuevo

Este cuaderno valida que el target `indice_exito` formal y sus categorías derivadas se comportan correctamente en el flujo completo de limpieza, split y evaluación.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.pipeline.streamview_success_target import compute_success_target, build_holdout_split

catalogo_path = project_root / 'data' / 'processed' / 'catalogo_streamview.csv'
train_path = project_root / 'data' / 'processed' / 'train_procesado.csv'
test_path = project_root / 'data' / 'processed' / 'test_procesado.csv'

print('Proyecto:', project_root)
df = pd.read_csv(catalogo_path, low_memory=False)
df = compute_success_target(df)

required = {'indice_exito', 'categoria_exito', 'target'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltan columnas: {sorted(missing)}')

assert df['indice_exito'].between(0, 1).all(), 'indice_exito fuera de rango [0,1]'
assert df['target'].equals(df['indice_exito']), 'target no coincide con indice_exito'

train_df, test_df = build_holdout_split(df)
assert len(train_df) + len(test_df) == len(df)
assert len(train_df) / len(df) >= 0.75 and len(test_df) / len(df) <= 0.25

print('Filas catalogo:', len(df))
print('Filas train:', len(train_df))
print('Filas test:', len(test_df))
print('Distribución por categoría:')
print(df['categoria_exito'].value_counts().to_string())
print('Rango indice_exito:', round(float(df['indice_exito'].min()), 4), 'a', round(float(df['indice_exito'].max()), 4))

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print('CSV de train/test exportados correctamente.')

Proyecto: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics
Filas catalogo: 32000
Filas train: 25600
Filas test: 6400
Distribución por categoría:
categoria_exito
Alto Éxito     10880
Medio Éxito    10560
Bajo Éxito     10560
Rango indice_exito: 0.0 a 0.7564
CSV de train/test exportados correctamente.
